In [9]:
require 'gd'

# this function show the image on the notebook
def display(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

# ═════════════════════════════════════════════════════════════════
#  Hist2D — 2D histogram renderer built on ruby-libgd
#
#  Usage:
#    hist = Hist2D.new(x: x_array, y: y_array)
#    hist.render("/work/jupyter-notebooks/hist2d.png")
#
#  Required:
#    x:        Array of Float
#    y:        Array of Float (same length as x)
#
#  Optional:
#    width:    canvas width  in pixels  (default: 500)
#    height:   canvas height in pixels  (default: 500)
#    bin_step: bin width                (default: 0.1)
#    bin_min:  left/bottom bin edge     (default: auto from data)
#    bin_max:  right/top bin edge       (default: auto from data)
#    xlim:     [min, max] display range (default: auto from data)
#    ylim:     [min, max] display range (default: auto from data)
#    bg:       [r,g,b] background       (default: white)
# ═════════════════════════════════════════════════════════════════
class Hist2D

  BLUES = [
    [247, 251, 255],
    [222, 235, 247],
    [198, 219, 239],
    [158, 202, 225],
    [107, 174, 214],
    [ 66, 146, 198],
    [ 33, 113, 181],
    [  8,  81, 156],
    [  8,  48, 107],
  ].freeze

  def initialize(
    x:,
    y:,
    width:    500,
    height:   500,
    bin_step: 0.1,
    bin_min:  nil,
    bin_max:  nil,
    xlim:     nil,
    ylim:     nil,
    bg:       [255, 255, 255]
  )
    raise ArgumentError, "x and y must have the same length" if x.size != y.size
    raise ArgumentError, "x and y cannot be empty"           if x.empty?

    @x       = x.map(&:to_f)
    @y       = y.map(&:to_f)
    @width   = width
    @height  = height
    @bin_step = bin_step.to_f
    @bg      = bg

    # Auto-compute bin range from data if not provided
    padding  = bin_step * 2
    @bin_min = (bin_min || [@x.min, @y.min].min - padding).to_f
    @bin_max = (bin_max || [@x.max, @y.max].max + padding).to_f

    # Auto-compute display limits from data if not provided
    @xlim = xlim || [@x.min, @x.max]
    @ylim = ylim || [@y.min, @y.max]
  end

  def render(path = "/work/jupyter-notebooks/hist2d.png")
    img   = new_image
    bins  = bin_edges
    nb    = bins.size - 1

    # ── Build 2D histogram ────────────────────────────────────────
    counts = Array.new(nb) { Array.new(nb, 0) }

    @x.zip(@y).each do |x, y|
      xi = bin_index(x, bins)
      yi = bin_index(y, bins)
      next if xi.nil? || yi.nil?
      counts[xi][yi] += 1
    end

    max_count = counts.flatten.max.to_f
    max_count = 1.0 if max_count.zero?

    # ── Pixel scale ───────────────────────────────────────────────
    x0, x1 = @xlim
    y0, y1 = @ylim
    scale_x = @width.to_f  / (x1 - x0)
    scale_y = @height.to_f / (y1 - y0)

    # ── Draw each bin ─────────────────────────────────────────────
    nb.times do |xi|
      nb.times do |yi|
        count = counts[xi][yi]
        next if count.zero?

        bx0 = bins[xi];  bx1 = bins[xi+1]
        by0 = bins[yi];  by1 = bins[yi+1]

        px0 = ((bx0 - x0) * scale_x).round
        px1 = ((bx1 - x0) * scale_x).round
        py0 = (@height - (by1 - y0) * scale_y).round
        py1 = (@height - (by0 - y0) * scale_y).round

        next if px1 <= 0 || px0 >= @width
        next if py1 <= 0 || py0 >= @height

        px0 = px0.clamp(0, @width-1);  px1 = px1.clamp(0, @width-1)
        py0 = py0.clamp(0, @height-1); py1 = py1.clamp(0, @height-1)

        t     = (count.to_f / max_count) ** 0.4
        color = colormap(t)
        img.filled_rectangle(px0, py0, px1, py1, GD::Color.rgb(*color))
      end
    end

    img.save(path)
  end

  private

  def bin_edges
    edges = []
    v = @bin_min
    while v <= @bin_max + 1e-9
      edges << v.round(10)
      v += @bin_step
    end
    edges
  end

  def bin_index(val, edges)
    return nil if val < edges.first || val >= edges.last
    lo = 0; hi = edges.size - 2
    while lo < hi
      mid = (lo + hi) / 2
      val < edges[mid+1] ? hi = mid : lo = mid + 1
    end
    lo
  end

  def colormap(t)
    t    = t.clamp(0.0, 1.0)
    idx  = t * (BLUES.size - 1)
    lo   = idx.floor.clamp(0, BLUES.size - 2)
    frac = idx - lo
    [0,1,2].map { |k|
      (BLUES[lo][k] + frac * (BLUES[lo+1][k] - BLUES[lo][k])).round.clamp(0, 255)
    }
  end

  def new_image
    img = GD::Image.new(@width, @height)
    img.filled_rectangle(0, 0, @width-1, @height-1, GD::Color.rgb(*@bg))
    img
  end
end

(irb):30: warning: already initialized constant #<Class:0x00007f4f653d1a60>::Hist2D::BLUES
(irb):30: warning: previous definition of BLUES was here


:new_image

In [10]:
require 'csv'
 
# ═════════════════════════════════════════════════════════════════
#  Generate sample weather CSV
#  Realistic correlation: humidity rises with temperature
#  in a tropical/humid climate range
#
#  temperature: 18°C .. 38°C  (mean 28, std 4)
#  humidity:    40% .. 95%    (correlated with temp + noise)
# ═════════════════════════════════════════════════════════════════
 
def randn(rng)
  u1 = rng.rand; u2 = rng.rand
  u1 = 1e-10 if u1.zero?
  Math.sqrt(-2.0 * Math.log(u1)) * Math.cos(2 * Math::PI * u2)
end
 
rng  = Random.new(42)
n    = 2000
 
rows = n.times.map do
  temp     = 28.0 + randn(rng) * 4.0          # mean 28°C, std 4
  humidity = 65.0 + 1.5 * (temp - 28.0) +
             randn(rng) * 8.0                  # correlated + noise
  temp     = temp.clamp(15.0, 42.0).round(1)
  humidity = humidity.clamp(30.0, 99.0).round(1)
  [temp, humidity]
end
 
CSV.open("/work/jupyter-notebooks/weather.csv", "w") do |csv|
  csv << ["temperature", "humidity"]
  rows.each { |row| csv << row }
end


[[33.3, 67.9], [32.3, 84.2], [26.9, 85.3], [28.6, 71.9], [21.9, 53.2], [30.5, 60.4], [29.1, 52.3], [31.9, 74.7], [36.9, 79.1], [33.0, 66.1], [19.8, 70.2], [24.6, 47.7], [29.7, 67.9], [29.1, 73.8], [30.9, 60.3], [27.3, 60.8], [21.9, 60.9], [37.1, 80.4], [33.2, 71.9], [30.6, 77.4], [26.4, 73.6], [25.2, 56.7], [26.1, 58.3], [25.3, 55.1], [23.9, 75.7], [21.1, 42.5], [28.0, 65.4], [34.1, 80.8], [28.5, 71.1], [29.0, 62.8], [30.1, 77.4], [21.8, 59.0], [15.4, 48.0], [23.7, 57.3], [26.7, 74.2], [28.0, 62.0], [34.2, 66.3], [26.3, 62.6], [20.2, 53.3], [28.3, 63.5], [24.2, 51.8], [32.5, 76.4], [19.5, 59.3], [28.7, 69.4], [25.4, 63.0], [32.4, 73.9], [24.5, 54.9], [28.2, 79.3], [26.8, 56.3], [30.6, 69.6], [31.3, 82.2], [32.0, 62.1], [34.9, 72.2], [28.6, 66.4], [27.9, 69.1], [24.7, 48.3], [28.2, 63.9], [26.7, 57.6], [23.8, 56.2], [34.3, 77.8], [29.3, 64.7], [38.4, 90.8], [28.7, 56.6], [27.1, 49.5], [27.5, 71.2], [31.9, 55.1], [27.9, 62.8], [26.5, 64.6], [29.8, 58.5], [26.4, 64.5], [27.0, 72.8], [32.1

In [11]:
# ═════════════════════════════════════════════════════════════════
#  Load weather.csv and render with Hist2D
#  Run generate_weather_csv.rb first to create the file
# ═════════════════════════════════════════════════════════════════

rows     = CSV.read("/work/jupyter-notebooks/weather.csv", headers: true)
x_data   = rows["temperature"].map(&:to_f)
y_data   = rows["humidity"].map(&:to_f)

puts "Loaded #{x_data.size} rows"
puts "Temperature range: #{x_data.min}°C .. #{x_data.max}°C"
puts "Humidity range:    #{y_data.min}% .. #{y_data.max}%"

Hist2D.new(
  x:        x_data,
  y:        y_data,
  width:    500,
  height:   500,
  bin_step: 0.5          # 0.5°C / 0.5% bins — good for this range
).render("/work/jupyter-notebooks/weather_hist2d.png")

display("weather_hist2d.png")

Loaded 2000 rows
Temperature range: 15.0°C .. 40.6°C
Humidity range:    33.2% .. 99.0%


"<img src='weather_hist2d.png' />"

In [15]:
rows   = CSV.read("weather.csv", headers: true)

name = "weather.png"
x_data = rows["temperature"].map(&:to_f)
y_data = rows["humidity"].map(&:to_f)
Hist2D.new(x: x_data, y: y_data).render("/work/jupyter-notebooks/#{name}")
display(name)

name = "auto.png"
# Auto limits — no xlim/ylim needed
Hist2D.new(x: x_data, y: y_data).render("/work/jupyter-notebooks/#{name}")
display(name)

"<img src='weather.png' />"

"<img src='auto.png' />"